# NMT Benchmarking, End-to-End Reproduction Notebook

**Indicative reference notebook** for reproducing the study. It documents the full
workflow in order; it is not a literal run log. Replace `YOUR_GH_TOKEN` and
`YOUR_HF_TOKEN` with your own tokens and never commit real tokens.

Systems: Google Translate (baseline) plus four evaluated systems, Amazon Translate,
Microsoft Translator, NLLB-200, and IndicTrans2 (six Indic languages only).
Corpus: 10,228 English UI strings, 12 target languages.

**Important:** the three model families use incompatible environments and must be run
in separate runtimes:
- NLLB-200: transformers 5.16.1 (GPU)
- IndicTrans2: transformers 4.46.1 + IndicTransToolkit (GPU, gated model)
- Metrics (COMET/BERTScore/CometKiwi): see the metrics environment; setuptools<81 required

Commercial systems (Google, Amazon, Microsoft) are API-based, run anywhere, and their
outputs are frozen snapshots at the recorded run dates.

## 0. Clone the repository
Replace the token; do not commit it. All later cells assume this working directory.

In [ ]:
%cd /content
!git clone https://YOUR_GH_TOKEN@github.com/Research-SoftwareLocalization/nmt-assistive-benchmarking.git
%cd nmt-assistive-benchmarking
!git config user.email "p20220505@hyderabad.bits-pilani.ac.in"
!git config user.name "Neeraj Kumar Sharma"

## 1. Translation, NLLB-200 (open-source, GPU)
Fresh GPU runtime. Do not pin transformers (Colab default works). Greedy decoding,
12 languages. Outputs to `data/nllb200/`; each file must have 10,228 lines.

In [ ]:
!pip install -q sentencepiece
import torch; print("CUDA:", torch.cuda.is_available())
!python scripts/run_all_nllb200.py
!pip freeze > requirements-lock.txt   # NLLB-200 environment record

## 2. Translation, IndicTrans2 (open-source, GPU, gated)
**Use a separate runtime.** IndicTrans2 needs transformers 4.46.1 + IndicTransToolkit,
which is incompatible with the NLLB environment. The model is gated: request access to
`ai4bharat/indictrans2-en-indic-1B` and log in first. Six Indic languages only.

In [ ]:
!pip install -q "transformers==4.46.1" IndicTransToolkit sentencepiece
# Runtime -> Restart session here so the pinned transformers loads.

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")
%cd /content/nmt-assistive-benchmarking
!python scripts/run_all_indictrans2.py
!pip freeze > requirements-indictrans2.txt   # IndicTrans2 environment record

## 3. Translation, commercial APIs (no GPU needed)
Amazon Translate (boto3, AWS creds) and Microsoft Translator (Azure key + region).
Both run anywhere; outputs are frozen snapshots at the run date. Google Translate
baseline was generated separately via the GOOGLETRANSLATE() Sheets function.

In [ ]:
# Amazon Translate
import os
os.environ["AWS_ACCESS_KEY_ID"] = "YOUR_AWS_KEY_ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_AWS_SECRET"
!pip install -q boto3 tqdm
!python scripts/run_all_amazon.py --workers 6

In [ ]:
# Microsoft Translator
import os
os.environ["AZURE_TRANSLATOR_KEY"] = "YOUR_AZURE_KEY"
os.environ["AZURE_TRANSLATOR_REGION"] = "eastus"
!pip install -q requests tqdm
!python scripts/run_all_microsoft.py

## 4. Integrity check + checksums
Every data file must be exactly 10,228 lines. Regenerate CHECKSUMS.txt whenever the
translation data changes (run on a machine with core.autocrlf=false).

In [ ]:
!for f in data/*/*.txt; do echo "$f: $(wc -l < $f)"; done
!md5sum data/source/en.txt data/nllb200/*.txt data/indictrans2/*.txt data/microsoft_translator/*.txt data/amazon_translate/*.txt > CHECKSUMS.txt
!wc -l CHECKSUMS.txt   # 43 files

## 5. Metrics (GPU)
One environment for all eight metrics. `setuptools<81` restores pkg_resources for the
COMET import chain. COMET and CometKiwi are gated, log in first. `sacrebleu[ja]` is
required for Japanese BLEU tokenization.

In [ ]:
!pip install -q "setuptools<81" sacrebleu "sacrebleu[ja]" nltk rouge-score bert-score unbabel-comet
import pkg_resources; print("pkg_resources OK")
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

In [ ]:
# Lexical (BLEU, METEOR, chrF++, ROUGE-L, TER)
!python scripts/compute_metrics.py --all --metrics lexical
# Neural (COMET, BERTScore)
!python scripts/compute_metrics.py --all --metrics neural
# Reference-free (CometKiwi); loads model once, saves incrementally, resumable
!python scripts/compute_cometkiwi.py --all

In [ ]:
# Merge + per-metric pivot tables, then capture the metrics environment
!python scripts/merge_metric_results.py
!python scripts/make_pivots.py
!pip freeze > requirements-metrics.txt

## 6. Commit results and environment records
Result files (not translation data), so no checksum regeneration is needed here.
Pass a fresh token inline; never commit a real token.

In [ ]:
!git add data/ metrics/ CHECKSUMS.txt requirements-lock.txt requirements-indictrans2.txt requirements-metrics.txt
!git commit -m "Add translations, metrics, and environment records"
!git push https://YOUR_GH_TOKEN@github.com/Research-SoftwareLocalization/nmt-assistive-benchmarking.git main

## Fair-comparison note
IndicTrans2 supports only the six Indic languages (hi, bn, mr, pa, ta, te). Compare it
with other systems only on those six; averaging it against systems evaluated on all
twelve languages is not a valid comparison.